# CelebA Multi-label Classification - GPU Accelerated (Full Dataset)

## Project Overview
This notebook contains the fully optimized, GPU-accelerated pipeline targeting the **full dataset**.

### 🏆 Level 1 & 2 Optimizations Triggered:
- ✅ **Resolution Upgrade:** 160x160 for improved detail recognition.
- ✅ **Class Imbalance Handling:** Dynamic `pos_weight` calculated and applied to BCE Loss.
- ✅ **Data Augmentation Expansion:** Horizontal Flip, Brightness, Contrast, and Rotation.
- ✅ **Base Threshold Shift:** Global target moved from 0.5 to 0.4 for improved recall.
- ✅ **Post-Training Threshold Tuning:** Automated grid search to maximize F1 Score.

### Architecture & Setup
- **Model**: SimpleCNN (4 block, 256 channels, GlobalAvgPool)
- **Hardware**: RTX 4070 Ti targeted (`cuda`), AMP enabled with Gradient Clipping
- **Data**: Full 162k CelebA partitions


In [ ]:
import os
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    print(f"[SEED] Random seed fixed to {seed} (cudnn.benchmark=True)")

set_seed(42)

# ── Config / Parameters ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[DEVICE] Hardware Targeted: {DEVICE}")

IMAGE_DIR      = r"C:\MLA\img_align_celeba"
ATTR_PATH      = r"C:\MLA\drive-download-20260318T181243Z-1-001\list_attr_celeba.txt"
PARTITION_PATH = r"C:\MLA\drive-download-20260318T181243Z-1-001\list_eval_partition.txt"

TRAIN_MODE     = "full"   

IMAGE_SIZE     = 160      # UPGRADED: 128 -> 160
BATCH_SIZE     = 128      # Optimized for 160x160 on 12GB VRAM
NUM_WORKERS    = 4        # Set to a stable value for Windows
PIN_MEMORY     = True

NUM_ATTRS               = 40
DROPOUT                 = 0.4
EPOCHS                  = 20
LEARNING_RATE           = 1e-3
EARLY_STOPPING_PATIENCE = 3
PREDICTION_THRESHOLD    = 0.4 

# Create directories if they don't exist
os.makedirs("outputs", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
print("[SETUP] Directories 'outputs' and 'checkpoints' verified.")


## 1. Dataframes Loading & Class Weight Calculation

In [ ]:
def load_dataframes():
    print("[DATA] Loading attribute and partition files...")
    if not os.path.exists(ATTR_PATH) or not os.path.exists(PARTITION_PATH):
        raise FileNotFoundError(f"Make sure {ATTR_PATH} and {PARTITION_PATH} exist.")
        
    attr = pd.read_csv(ATTR_PATH, sep=r'\s+', header=1, index_col=0)
    attr.index.name = 'image_id'
    attr = ((attr + 1) // 2).reset_index()
    
    splits = pd.read_csv(PARTITION_PATH, sep=' ', header=None, names=['image_id', 'split'])
    df = splits.merge(attr, on='image_id')
    
    train_df = df[df['split'] == 0].drop('split', axis=1).reset_index(drop=True)
    val_df   = df[df['split'] == 1].drop('split', axis=1).reset_index(drop=True)
    test_df  = df[df['split'] == 2].drop('split', axis=1).reset_index(drop=True)
    
    if TRAIN_MODE == "subset":
        train_df = train_df.iloc[:50000].reset_index(drop=True)
    
    print(f"[DATA] Mode = {TRAIN_MODE} -> {len(train_df):,} training images | {len(val_df):,} val images")
    return train_df, val_df, test_df

train_df, val_df, test_df = load_dataframes()

print("\n[CLASS WEIGHTS] Computing `pos_weight` for rare attributes...")
attr_cols = [c for c in train_df.columns if c != 'image_id']
y_train = train_df[attr_cols].values
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
pos_weight_tensor = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32)
pos_weight_tensor = torch.clamp(pos_weight_tensor, max=10.0) 
print(f"[CLASS WEIGHTS] Pos_weights computed (max clamped to 10.0)")


## 2. Dataset & Dataloaders

In [ ]:
def to_tensor_normalised(img: Image.Image) -> torch.Tensor:
    arr = np.array(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1)
    return (tensor - 0.5) / 0.5

def train_transform(img: Image.Image) -> torch.Tensor:
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    if random.random() < 0.5: img = ImageOps.mirror(img)  
    if random.random() < 0.5: 
        enhancer = ImageEnhance.Brightness(img)
        img = enhancer.enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.5: 
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.5: 
        img = img.rotate(random.uniform(-10, 10)) 
    return to_tensor_normalised(img)

def eval_transform(img: Image.Image) -> torch.Tensor:
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    return to_tensor_normalised(img)

class CelebADataset(Dataset):
    def __init__(self, df, img_dir, is_train=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.is_train = is_train
        self.attr_cols = [c for c in df.columns if c != 'image_id']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            img_id = row['image_id']
            img_path = os.path.join(self.img_dir, img_id)
            img = Image.open(img_path).convert('RGB')
            img_tensor = train_transform(img) if self.is_train else eval_transform(img)
            label = torch.tensor(row[self.attr_cols].values.astype(float), dtype=torch.float32)
            return img_tensor, label
        except Exception as e:
            # Return placeholder if image load fails to keep training going
            print(f" Error loading {img_id}: {e}")
            return torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE)), torch.zeros(len(self.attr_cols))

    def get_attr_names(self):
        return self.attr_cols

def get_dataloaders(train_df, val_df, test_df):
    if not os.path.exists(IMAGE_DIR):
        raise FileNotFoundError(f"IMAGE_DIR not found at {IMAGE_DIR}")
        
    train_ds = CelebADataset(train_df, IMAGE_DIR, is_train=True)
    val_ds   = CelebADataset(val_df,   IMAGE_DIR, is_train=False)
    test_ds  = CelebADataset(test_df,  IMAGE_DIR, is_train=False)

    loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    if NUM_WORKERS > 0:
        loader_kwargs['persistent_workers'] = True
        loader_kwargs['prefetch_factor'] = 2

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader, train_ds.get_attr_names()

train_loader, val_loader, test_loader, attr_names = get_dataloaders(train_df, val_df, test_df)
print(f"[DATA] Dataloaders ready. Workers: {NUM_WORKERS}")


## 3. GPU-Optimized SimpleCNN

In [ ]:
def _conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=40, dropout=0.4):
        super().__init__()
        self.block1 = _conv_block(3,   64)
        self.block2 = _conv_block(64,  128)
        self.block3 = _conv_block(128, 256) 
        self.block4 = _conv_block(256, 256)
        self.pool    = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout)
        self.fc      = nn.Linear(256, num_classes)
        self._init_weights()

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

model = SimpleCNN(num_classes=NUM_ATTRS, dropout=DROPOUT).to(DEVICE)
print(f"\n[MODEL] Initialization successful. Params: {sum(p.numel() for p in model.parameters()):,}")


## 4. Evaluators & Training Setup

In [ ]:
def compute_metrics(preds_binary, labels):
    # preds_binary, labels are tensors on same device
    tp = (preds_binary * labels).sum(dim=0)
    fp = (preds_binary * (1 - labels)).sum(dim=0)
    fn = ((1 - preds_binary) * labels).sum(dim=0)
    per_label_acc = (preds_binary == labels).float().sum(dim=0) / labels.shape[0]
    accuracy = per_label_acc.mean().item()
    per_precision = tp / (tp + fp + 1e-8)
    per_recall    = tp / (tp + fn + 1e-8)
    precision = per_precision.mean().item()
    recall    = per_recall.mean().item()
    f1        = (2 * precision * recall / (precision + recall + 1e-8))
    return {'accuracy': accuracy*100, 'precision': precision*100, 'recall': recall*100, 'f1': f1*100}

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(DEVICE))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None


## 5. Main Training Loop

In [ ]:
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, leave=False, desc="[Train]" if training else "[Val  ]")
    
    for images, labels in pbar:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(scaler is not None)):
                outputs = model(images)
                loss = criterion(outputs, labels)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        else:
            with torch.no_grad(), torch.cuda.amp.autocast(enabled=(scaler is not None)):
                outputs = model(images)
                loss = criterion(outputs, labels)
        
        total_loss += loss.item()
        preds_binary = (torch.sigmoid(outputs) > PREDICTION_THRESHOLD).float()
        all_preds.append(preds_binary.cpu())
        all_labels.append(labels.cpu())
        
    avg_loss = total_loss / len(loader)
    metrics = compute_metrics(torch.cat(all_preds), torch.cat(all_labels))
    return avg_loss, metrics

train_losses, val_losses, val_accuracies, val_f1s = [], [], [], []
best_val_loss = float('inf')
patience_count = 0

print(f"\n[TRAINING] Starting {EPOCHS} epochs on {DEVICE}...")
for epoch in range(EPOCHS):
    start_t = time.time()
    train_loss, train_m = run_epoch(train_loader, training=True)
    val_loss, val_m     = run_epoch(val_loader, training=False)
    epoch_time = time.time() - start_t
    
    train_losses.append(train_loss); val_losses.append(val_loss)
    val_accuracies.append(val_m['accuracy']); val_f1s.append(val_m['f1'])
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} [{epoch_time:.0f}s] | Loss: T={train_loss:.4f} V={val_loss:.4f} | Acc: {val_m['accuracy']:.2f}% F1: {val_m['f1']:.2f}%")
    
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss; patience_count = 0
        torch.save(model.state_dict(), "checkpoints/best_model_gpu_full.pth")
    else:
        patience_count += 1
        if patience_count >= EARLY_STOPPING_PATIENCE: 
            print(f"[EARLY STOP] Patience reached at epoch {epoch+1}")
            break


## 6. Training Plot Evaluation

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1); plt.plot(train_losses, 'o-', label='Train'); plt.plot(val_losses, 'o-', label='Val'); plt.title('Loss'); plt.xlabel('Epoch'); plt.legend()
plt.subplot(1, 2, 2); plt.plot(val_accuracies, 'o-', label='Acc'); plt.plot(val_f1s, 's-', label='F1'); plt.title('Metrics'); plt.xlabel('Epoch'); plt.legend()
plt.savefig("outputs/training_plots.png")
plt.show()


## 7. Post-Training Threshold Tuning

In [ ]:
print("\n[TUNING] Sweeping thresholds on Validation data...")
if os.path.exists("checkpoints/best_model_gpu_full.pth"):
    model.load_state_dict(torch.load("checkpoints/best_model_gpu_full.pth"))
model.eval()

all_val_probs, all_val_labels = [], []
with torch.no_grad(), torch.cuda.amp.autocast(enabled=(scaler is not None)):
    for images, labels in tqdm(val_loader):
        outputs = model(images.to(DEVICE))
        all_val_probs.append(torch.sigmoid(outputs).cpu()); all_val_labels.append(labels.cpu())

all_val_probs, all_val_labels = torch.cat(all_val_probs), torch.cat(all_val_labels)
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]:
    m = compute_metrics((all_val_probs > t).float(), all_val_labels)
    print(f"T [{t:.2f}] -> F1: {m['f1']:5.2f}% | Rec: {m['recall']:5.2f}% | Prec: {m['precision']:5.2f}%")
